In [1]:
import sys
dice_path    = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE"
dice_x_path  = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"

# insert at position 0 so they shadow anything of the same name in
# site-packages; if you’d rather keep PyPI packages as the default, use
# sys.path.append(…) instead.
for p in (dice_x_path, dice_path):
    if p not in sys.path:
        sys.path.insert(0, p)

In [2]:
import dice_ml
from dice_ml.utils import helpers, neuralnetworks
import dice_ml_x
from dice_ml_x.utils import helpers, neuralnetworks
import pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import itertools
from random import randrange
import random
import os
from collections import OrderedDict
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import KNeighborsClassifier
from typing import Optional, Union

import numpy as np, pandas as pd, random, torch, tensorflow as tf
from collections import OrderedDict, defaultdict
from pathlib import Path
from tqdm import tqdm
from scipy.stats import ttest_rel

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
with open('final_benchmarking_results.pkl', 'rb') as res_file:
    dice_benchmarking_results = pickle.load(res_file)

In [5]:
with open('/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X/docs/source/notebooks/benchmarking_results_23_01_2025-01_05.pkl', 'rb') as res_file:
    dice_x_benchmarking_results = pickle.load(res_file)

In [6]:
backends = ['sklearn', 'PYT', 'TF2']
def load_torch_model(model_path, in_features):
    dummy_state_dict = torch.load(model_path)
    dummy_state_dict = {f'model.{key}': value for key, value in dummy_state_dict.items()}
    model = neuralnetworks.PYTModel(in_features)
    model.load_state_dict(dummy_state_dict)
    return model

def load_tensorflow_model(model_path):
    model = neuralnetworks.TF2Model()
    model.load_weights(model_path)
    return model

dataset_names = [
    "compas-recidivism",
    "adult-income",
    "lending-club",
    "german-credit"
]
dice_models = {}
for name in dataset_names:
    dice_models[name] = {}
    for backend in backends:
        if backend == 'sklearn':
            dice_models[name][backend] = dice_benchmarking_results[name][backend]['model']
        elif backend == 'PYT':
            model_path = dice_benchmarking_results[name][backend]['model_path']
            num_features = dice_benchmarking_results[name][backend]['metrics']['num_features']
            dice_models[name][backend] = load_torch_model(model_path, num_features)
        elif backend == 'TF2':
            model_path = dice_benchmarking_results[name][backend]['model_path']
            dice_models[name][backend] = load_tensorflow_model(model_path)

/var/folders/8m/6z48y8212x782rk9dpgmm9lm0000gn/T/ipykernel_23479/1462205730.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dummy_state_dict = torch.load(model_path)


In [7]:

dice_x_models = {}
for name in dataset_names:
    dice_x_models[name] = {}
    for backend in backends:
        if backend == 'sklearn':
            dice_x_models[name][backend] = dice_x_benchmarking_results[name][backend]['model']
        elif backend == 'PYT':
            model_path = dice_x_benchmarking_results[name][backend]['model_path']
            num_features = dice_x_benchmarking_results[name][backend]['metrics']['num_features']
            dice_x_models[name][backend] = load_torch_model(model_path, num_features)
        elif backend == 'TF2':
            model_path = dice_x_benchmarking_results[name][backend]['model_path']
            dice_x_models[name][backend] = load_tensorflow_model(model_path)

/var/folders/8m/6z48y8212x782rk9dpgmm9lm0000gn/T/ipykernel_23479/1462205730.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dummy_state_dict = torch.load(model_path)


In [8]:
# metric functions
def compute_validity(exp) -> float:
    return exp.get_validity_percentage()

def compute_mad(data_class: dice_ml.Data, normalized=False) -> dict:
    return data_class.get_valid_mads(normalized=normalized)

def compute_continuous_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    mads = compute_mad(data_class)
    total_proximity = 0.0

    for feature in data_class.continuous_feature_names:
        diff = np.abs(C[feature] - x[feature].iloc[0])
        total_proximity += diff / mads[feature]

    return -np.mean(total_proximity)

def compute_continuous_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    mads = compute_mad(data_class)
    normalized_diff = (np.abs(C[data_class.continuous_feature_names] - x[data_class.continuous_feature_names].iloc[0]) / 
                      np.array([mads[feature] for feature in data_class.continuous_feature_names]).reshape(1, -1))
    mean_per_CF = np.nanmean(normalized_diff, axis=1)
    return -np.mean(mean_per_CF)

def compute_categorical_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    categorical_feats = data_class.categorical_feature_names
    if len(x) == 0 or len(categorical_feats) == 0:
        return 0.0
    x_values: pd.Series = x.iloc[0]
    diff_matrix: pd.DataFrame = C[categorical_feats] != x_values[categorical_feats]
    diff_count: pd.Series = diff_matrix.sum(axis=1)
    d_cat = len(categorical_feats)

    average_distance = diff_count.mean() / (d_cat * len(C)) if d_cat > 0 else 0.0

    return 1 - average_distance

def compute_continuous_diversity(C: pd.DataFrame, data_class: dice_ml.Data) -> float:
    cont_feats = data_class.continuous_feature_names

    if isinstance(C, pd.DataFrame):
        X = C[cont_feats].values
    else:
        X = C

    k, d = X.shape

    if k < 2 or d == 0:
        return 0.0
    
    mad_dict = compute_mad(data_class)
    mad_vector = np.array([
        mad_dict[feature] if mad_dict[feature] != 0 else 1.0
        for feature in cont_feats
    ]).reshape(1, d)
    
    diff = np.abs(X[:, None, :] - X[None, :, :])
    normalized_diff = diff / mad_vector

    pairwise_dist = np.mean(normalized_diff, axis=2)
    triu_indices = np.triu_indices(k, k=1)
    if len(triu_indices[0]) == 0:
        return 0.0
    average_distance = np.mean(pairwise_dist[triu_indices])
    return average_distance

def compute_continuous_diversity(C: pd.DataFrame, data_class: dice_ml.Data) -> float:
    mads = compute_mad(data_class)  
    X = C[data_class.continuous_feature_names].values
    diff = np.abs(X[:, None, :] - X[None, :, :])
    normalized_diff = diff / np.array([mads[feature] for feature in data_class.continuous_feature_names]).reshape(1, 1, -1)
    pairwise_dist = np.nanmean(normalized_diff, axis=2)
    triu_indices = np.triu_indices(len(C), k=1)
    return np.mean(pairwise_dist[triu_indices]) if len(triu_indices[0]) > 0 else 0.0

def compute_categorical_diversity(C: pd.DataFrame, data_class: dice_ml.Data) -> float:
    cat_feats = data_class.categorical_feature_names

    if isinstance(C, pd.DataFrame):
        X = C[cat_feats].values
    else:
        X = C
    
    k, d = X.shape

    if k < 2 or d == 0:
        return 0.0

    diff = (X[:, None, :] != X[None, :, :]).astype(np.float32)

    pairwise_dist = np.mean(diff, axis=2)
    triu_indices = np.triu_indices(k, k=1)
    if len(triu_indices[0]) == 0:
        return 0.0
    average_distance = np.mean(pairwise_dist[triu_indices])
    return average_distance

def compute_count_diversity(C: pd.DataFrame):
    if isinstance(C, pd.DataFrame):
        X = C.values
    else:
        X = C

    k, d = X.shape

    if X.size == 0 or k < 2 or d == 0:
        return 0.0
    
    diff = (X[:, None, :] != X[None, :, :]).astype(np.float32)
    triu_indices = np.triu_indices(k, k=1)
    total_diff = np.sum(diff[triu_indices[0], triu_indices[1], :])

    n_pairs = len(triu_indices[0])
    return total_diff / (n_pairs * d)

def compute_sparsity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    cont_feats = data_class.continuous_feature_names
    cont_CFs = C[cont_feats].to_numpy()
    cont_X = x[cont_feats].to_numpy()

    k, d = cont_CFs.shape

    diff = (cont_CFs != cont_X[0])

    num_changed = diff.sum()

    return 1 - (num_changed / (k * d))


def robustness_flip_rate(
    C: pd.DataFrame,
    target_col: str,
    data_iface: "dice_ml.Data",
    backend: str,
    model,
    *,
    noise_sd: float = 0.10,        # 10 % of feature range, per feature
    cat_flip_p: float = 0.20,      # 20 % chance to flip a categorical value
    n_repeat: int = 50,            # Monte-Carlo samples per CF
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Robustness =  (#perturbed samples that KEEP the original class)
/ (total #perturbed samples)

    ↪ 1 = fully stable 0 = always flips
    """
    rng = rng or np.random.default_rng()

    # ------------------------------------------------------------------ helpers
    def _predict_class(X_raw: pd.DataFrame) -> np.ndarray:
        """Return class label 0/1 for X_raw (unencoded)."""
        if backend == "sklearn":
            return model.predict(X_raw)

        X_enc = data_iface.get_ohe_min_max_normalized_data(X_raw).values
        if backend == "PYT":
            import torch
            logits = model(torch.tensor(X_enc, dtype=torch.float32)
                          ).detach().cpu().numpy().ravel()
        elif backend == "TF2":
            import tensorflow as tf
            logits = model.predict(tf.constant(X_enc, dtype=tf.float32)).ravel()
        else:
            raise ValueError(f"Unknown backend {backend}")
        return (logits >= 0.5).astype(int)

    # ------------------------------------------------------------------ original predictions
    X_raw = C.drop(columns=[target_col]).reset_index(drop=True)
    y_orig = _predict_class(X_raw)

    # continuous & categorical bookkeeping
    cont_cols = data_iface.continuous_feature_names
    cat_cols  = data_iface.categorical_feature_names
    ranges    = data_iface.get_features_range_float()[1]          # {col: (lo, hi)}
    cat_vals  = {c: data_iface.get_features_range()[1][c] for c in cat_cols}

    n_cf   = len(C)
    n_mc   = n_repeat
    n_tot  = n_cf * n_mc
    kept   = 0

    # ------------------------------------------------------------------ Monte-Carlo loop
    for _ in range(n_repeat):
        X_noisy      = X_raw.copy()

        # ----- continuous perturbations
        for col in cont_cols:
            lo, hi = ranges[col]
            span   = hi - lo
            X_noisy[col] = np.clip(
                X_noisy[col] + rng.normal(0, noise_sd * span, size=n_cf),
                lo, hi
            )

        # ----- categorical flips (convert to object to avoid pandas category error)
        for col in cat_cols:
            X_noisy[col] = X_noisy[col].astype(object)
            mask = rng.random(n_cf) < cat_flip_p
            if mask.any():
                X_noisy.loc[mask, col] = rng.choice(cat_vals[col], size=mask.sum())

        # ----- predict & compare
        y_noisy = _predict_class(X_noisy)
        kept   += np.sum(y_noisy == y_orig)

    robustness = kept / n_tot
    return float(robustness)


def one_nn_fidelity(
    x_df: pd.DataFrame,
    cfs_df: pd.DataFrame,
    data_interface,
    model,
    backend: str,              # "sklearn" | "PYT" | "TF2"
    radius_mad: float,
    n_samples: int = 1000,
    rng: Optional[np.random.Generator] = None,
) -> float:

    rng = rng or np.random.default_rng()
    OUTCOME = data_interface.outcome_name            # e.g. "income"

    # ------------------------------------------------------------------
    # helper → class label 0/1   (always after dropping OUTCOME col)
    # ------------------------------------------------------------------
    def predict_classes(df_raw: pd.DataFrame) -> np.ndarray:
        X_raw = df_raw.drop(columns=[OUTCOME], errors="ignore")

        if backend == "sklearn":
            out = model.predict(X_raw)
            if out.ndim == 2 or (out.ndim == 1 and np.issubdtype(out.dtype, np.floating)):
                out = (out >= 0.5).astype(int).ravel()
            return out.astype(int)

        enc = data_interface.get_ohe_min_max_normalized_data(X_raw).values
        if backend == "PYT":
            import torch
            preds = model(torch.tensor(enc, dtype=torch.float32)).detach().cpu().numpy().ravel()
        elif backend == "TF2":
            import tensorflow as tf
            preds = model.predict(tf.constant(enc, dtype=tf.float32)).ravel()
        else:
            raise ValueError(f"Unknown backend {backend}")
        return (preds >= 0.5).astype(int)

    # ------------------------------------------------------------------
    # 1) build 1-NN proxy
    # ------------------------------------------------------------------
    train_raw = pd.concat([x_df, cfs_df], ignore_index=True).drop(columns=[OUTCOME], errors="ignore")
    y_train   = predict_classes(train_raw)

    if backend == "sklearn":
        train_cat = train_raw.copy()
        for col in data_interface.categorical_feature_names:
            train_cat[col] = train_cat[col].astype("category")

        X_train_df = pd.get_dummies(train_cat)
        X_train_df = X_train_df.fillna(X_train_df.mean())
        dummy_cols = X_train_df.columns.to_list()
        X_train    = X_train_df.values
    else:
        X_train = data_interface.get_ohe_min_max_normalized_data(train_raw).values
        dummy_cols = None

    knn = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)

    # ------------------------------------------------------------------
    # 2) sample synthetic points (outcome col never added)
    # ------------------------------------------------------------------
    cont_cols = data_interface.continuous_feature_names
    cat_cols  = data_interface.categorical_feature_names
    mad_vals  = data_interface.get_valid_mads()
    cat_levels = {col: data_interface.get_features_range()[1][col] for col in cat_cols}

    synth_rows = []
    for _ in range(n_samples):
        row = x_df.drop(columns=[OUTCOME], errors="ignore").copy()
        for col in cont_cols:
            row.at[row.index[0], col] += rng.uniform(-mad_vals[col]*radius_mad,
                                                     mad_vals[col]*radius_mad)
        for col in cat_cols:
            row.at[row.index[0], col] = rng.choice(cat_levels[col])
        synth_rows.append(row)
    synth_raw = pd.concat(synth_rows, ignore_index=True)

    # ------------------------------------------------------------------
    # 3) encode synth same way as train
    # ------------------------------------------------------------------
    if backend == "sklearn":
        synth_cat = synth_raw.copy()
        for col in cat_cols:
            synth_cat[col] = synth_cat[col].astype("category")
        X_synth_df = pd.get_dummies(synth_cat)
        for col in dummy_cols:
            if col not in X_synth_df.columns:
                X_synth_df[col] = 0
        X_synth = X_synth_df[dummy_cols].fillna(X_train_df.mean()).values
    else:
        X_synth = data_interface.get_ohe_min_max_normalized_data(synth_raw).values

    # ------------------------------------------------------------------
    # 4) fidelity score
    # ------------------------------------------------------------------
    orig_pred  = predict_classes(synth_raw)
    proxy_pred = knn.predict(X_synth)
    return float(np.mean(orig_pred == proxy_pred))



In [ ]:
# -----------------------------------------------------
# Config
# -----------------------------------------------------
DATASETS   = [
    (helpers.load_compas_dataset(),  "twoyearrecid", "compas-recidivism"),
    (helpers.load_adult_income_dataset(), "income",  "adult-income"),
    (helpers.load_lending_club_dataset(), "loan_status", "lending-club"),
    (helpers.load_german_credit_dataset(), "credit_risk", "german-credit")
]
BACKENDS   = ["sklearn", "PYT", "TF2"]
CF_METHODS = {
    "DiCE-Ext": dict(total_CFs=10, diversity_weight=1.0, robustness_weight=0.4, max_iter=500),
    "DiCE":     dict(total_CFs=10, diversity_weight=1.0)
}
N_TEST_POINTS = 50
K_CFS         = 5
RANDOM_SEED   = 42
OUT_DIR       = Path("cf_metric_results_robustness"); OUT_DIR.mkdir(exist_ok=True)
RADIUS_SET = [0.5, 1.0, 2.0]

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# -----------------------------------------------------
# Utility to collect metrics for ONE (dataset, backend, method)
# -----------------------------------------------------
def eval_method(df, target, name, backend, method_label, exp_opts):
    RAW_DIR = OUT_DIR / "raw_vectors"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    target_col  = df[target]
    train_df, test_df = train_test_split(df, test_size=0.2,
                                         random_state=RANDOM_SEED, stratify=target_col)
    

    opts = exp_opts.copy()

    if backend == "sklearn":
        opts["maxiterations"] = opts.pop("max_iter", 500)
    else:
        opts.pop("maxiterations", None)

    if method_label == "DiCE-Ext":
        exp_module = dice_ml_x
        models = dice_x_models
    else:
        exp_module = dice_ml
        models = dice_models

    d = exp_module.Data(
            dataframe=train_df,
            continuous_features=list(train_df.select_dtypes(include=np.number).columns.difference([target])),
            outcome_name=target)


    model_opts = OrderedDict(model=models[name][backend], backend=backend)
    method = 'genetic' if backend == 'sklearn' else 'gradient'

    if backend != 'sklearn':
        model_opts['func'] = 'ohe-min-max'
        if backend == 'PYT':
            model_opts['model'] = models[name][backend].model

    m = exp_module.Model(**model_opts)
    exp = exp_module.Dice(d, m, method=method)

    # sample test points once
    sampled_idx = np.random.choice(len(test_df), N_TEST_POINTS, replace=False)
    metrics = defaultdict(list)

    for idx in tqdm(sampled_idx, desc=f"{name}-{backend}-{method_label}"):
        x_full  = test_df.iloc[idx:idx+1]
        x_query = x_full.drop(columns=[target])

        
        dice_exp = exp.generate_counterfactuals(x_query, **opts)
        C        = dice_exp.to_dataframe()
        '''except Exception as e:
            # log and skip this point
            print(f"[warn] {name}-{backend}-{method_label}: "
                f"no CFs for idx={idx} ({e})")
            continue'''

        if C.empty:
            print(f"[warn] {name}-{backend}-{method_label}: "
                f"empty CF set for idx={idx}")
            continue
        
        # fix NaNs / recast categorical where needed
        na_cols = C.columns[C.isna().any()]
        C[na_cols] = C[na_cols].fillna(x_query[na_cols].iloc[0])

        # ---- metric suite ----
        vld   = compute_validity(exp)
        prox  = (compute_continuous_proximity(C, x_full, d) +
                 compute_categorical_proximity(C, x_full, d))
        spars = compute_sparsity(C, x_full, d)
        cdiv  = compute_continuous_diversity(C, d) + compute_categorical_diversity(C, d)
        stab  = robustness_flip_rate(C, target, d, backend, m.model)

        for r in RADIUS_SET:
            f1 = one_nn_fidelity(
                    x_query, C, d, m.model,
                    backend=backend,
                    radius_mad=r,
                    n_samples=1000
                )
            metrics[f"fidelity_1nn_{r}"].append(f1)

        metrics["validity"].append(vld)
        metrics["proximity"].append(prox)
        metrics["sparsity"].append(spars)
        metrics["diversity"].append(cdiv)
        metrics["robustness"].append(stab)

    # aggregate
    out = {f"{k}_mean":  np.mean(v) for k, v in metrics.items()}
    out.update({f"{k}_sd": np.std(v, ddof=1) for k, v in metrics.items()})
    out.update(dict(dataset=name, backend=backend, method=method_label, n=N_TEST_POINTS))

    row_id = f"{name}__{backend}__{method_label}__{N_TEST_POINTS}"
    raw_path = RAW_DIR / f"{name}__{backend}__{method_label}.npz"
    if raw_path.exists():
        return None, None
    
    np.savez_compressed(
        RAW_DIR / f"{row_id}.npz",
        **{k: np.asarray(v, dtype=float) for k, v in metrics.items()}
    )
    return out, metrics


# -----------------------------------------------------
# MAIN LOOP
# -----------------------------------------------------
OUT_DIR = Path("cf_metric_results_robustness"); OUT_DIR.mkdir(exist_ok=True)

def summary_path(ds: str, backend: str, method: str) -> Path:
    return OUT_DIR / f"{ds}__{backend}__{method}.csv"

rows, raw_metrics = [], {}

for df, target, ds_name in DATASETS:
    for backend in BACKENDS:
        for method_label, opts in CF_METHODS.items():

            this_path = summary_path(ds_name, backend, method_label)

            # ---------- skip if already computed ----------
            if this_path.exists():
                print(f"[skip] {ds_name}-{backend}-{method_label} "
                      f"(cached at {this_path.name})")
                cached_row = pd.read_csv(this_path).iloc[0].to_dict()
                rows.append(cached_row)
                continue

            # ---------- run normally ----------
            res_row, per_point = eval_method(
                    df, target, ds_name, backend, method_label, opts)

            # save immediately
            pd.DataFrame([res_row]).to_csv(this_path, index=False)
            print(f"[saved] {this_path.name}")

            rows.append(res_row)
            raw_metrics[(ds_name, backend, method_label)] = per_point


summary_files = list(OUT_DIR.glob("*.csv"))
result_df = pd.concat([pd.read_csv(p) for p in summary_files],
                      ignore_index=True)
result_df.to_csv(OUT_DIR / "summary_metrics_all.csv", index=False)

print("\n=== Aggregate results ===")
'''print(result_df[["dataset","backend","method",
                 "validity_mean","proximity_mean",
                 "sparsity_mean","diversity_mean","robustness_mean",
                 "fidelity_1nn_0.5_mean", "fidelity_1nn_1.0_mean",
                 "fidelity_1nn_2.0_mean"]])'''

print(result_df[["dataset","backend","method",
                 "robustness_mean","robustness_sd"]])

compas-recidivism-sklearn-DiCE-Ext: 100%|██████████| 20/20 [08:31<00:00, 25.58s/it]


[saved] compas-recidivism__sklearn__DiCE-Ext.csv


compas-recidivism-sklearn-DiCE: 100%|██████████| 20/20 [01:07<00:00,  3.39s/it]


[saved] compas-recidivism__sklearn__DiCE.csv


100%|██████████| 1/1 [00:04<00:00,  4.12s/it]  | 0/20 [00:00<?, ?it/s]

...

1/1 [==============================] - 0s 6ms/step


german-credit-TF2-DiCE:  95%|█████████▌| 19/20 [16:58<00:53, 53.34s/it]WARNING:root: MAD for feature number_of_existing_credits_at_this_bank is 0, so replacing it with 1.0 to avoid error.


1/1 [==============================] - 0s 6ms/step


german-credit-TF2-DiCE: 100%|██████████| 20/20 [18:01<00:00, 54.07s/it]

[saved] german-credit__TF2__DiCE.csv

=== Aggregate results ===
              dataset  backend    method  robustness_mean  robustness_sd
0   compas-recidivism  sklearn  DiCE-Ext         0.808480       0.038225
1        adult-income  sklearn      DiCE         0.704600       0.059266
2   compas-recidivism      PYT      DiCE         0.922622       0.043473
3        lending-club      PYT      DiCE         0.589479       0.044475
4        adult-income      PYT  DiCE-Ext         0.867400       0.043679
5        lending-club  sklearn      DiCE         0.575300       0.030548
6        lending-club      PYT  DiCE-Ext         0.587973       0.054569
7        lending-club  sklearn  DiCE-Ext         0.607500       0.029607
8   compas-recidivism      TF2  DiCE-Ext         0.900000       0.013107
9       german-credit      TF2      DiCE         0.916900       0.022105
10       adult-income      PYT      DiCE         0.853600       0.052490
11      german-credit      TF2  DiCE-Ext         0.963500   

In [12]:
METRIC_KEYS = ["validity", "proximity", "sparsity",
               "diversity", "robustness", "fidelity_1nn_0.5",
               "fidelity_1nn_1.0", "fidelity_1nn_2.0"]

print("\n=== Paired t-tests:  DiCE-Ext  vs  DiCE ===")
for _, _, ds_name in DATASETS:
    for be in BACKENDS:
        print(f"\n{ds_name}  |  backend = {be}")
        for key in METRIC_KEYS:
            A = np.array(raw_metrics[(ds_name, be, "DiCE-Ext")][key])
            B = np.array(raw_metrics[(ds_name, be, "DiCE")][key])
            if len(A) == 0 or len(B) == 0:
                print(f"  {key:<10}:  n/a   (missing data)")
                continue
            p = ttest_rel(A, B).pvalue
            print(f"  {key:<10}:  p = {p:0.4f}")


=== Paired t-tests:  DiCE-Ext  vs  DiCE ===

compas-recidivism  |  backend = sklearn


KeyError: ('compas-recidivism', 'sklearn', 'DiCE-Ext')

In [14]:
from typing import Dict

RAW_DIR      = Path("cf_metric_results_robustness/raw_vectors")
DATASETS     = ["adult-income", "compas-recidivism",
                "german-credit", "lending-club"]
BACKENDS     = ["sklearn", "PYT", "TF2"]
'''METRIC_KEYS  = ["validity", "proximity", "sparsity",
                "diversity", "robustness",
                "fidelity_1nn_0.5", "fidelity_1nn_1.0", "fidelity_1nn_2.0"]'''

METRIC_KEYS  = ["robustness"]

def load_npz(path: Path) -> Dict[str, np.ndarray]:
    """Load npz → dict of metric→vector (shape = n_test_points,)"""
    with np.load(path, allow_pickle=False) as npz:
        return {k: npz[k].astype(float) for k in METRIC_KEYS if k in npz}

results = []

for ds in DATASETS:
    for be in BACKENDS:
        f_base = RAW_DIR / f"{ds}__{be}__DiCE__20.npz"
        f_ext  = RAW_DIR / f"{ds}__{be}__DiCE-Ext__20.npz"
        if not (f_base.exists() and f_ext.exists()):
            print(f"[skip] raw vectors missing for {ds}/{be}")
            continue

        vec_A = load_npz(f_ext)      # “DiCE-Ext”
        vec_B = load_npz(f_base)     # plain “DiCE”

        row = {"dataset": ds, "backend": be}
        for key in METRIC_KEYS:
            a, b = vec_A[key], vec_B[key]
            # ensure equal length & no-nan
            m = (~np.isnan(a)) & (~np.isnan(b))
            if m.sum() < 2:          # need ≥2 pairs for t-test
                row[f"{key}_p"] = np.nan
                continue
            p_val = ttest_rel(a[m], b[m]).pvalue   # two-sided by default
            row[f"{key}_p"] = p_val
        results.append(row)

pvals_df = pd.DataFrame(results).set_index(["dataset","backend"])
print(pvals_df.round(4))
pvals_df.to_csv("cf_metric_results_robustness/cf_metric_pvals.csv")

                           robustness_p
dataset           backend              
adult-income      sklearn        0.3465
                  PYT            0.3599
                  TF2            0.0025
compas-recidivism sklearn        0.1129
                  PYT            0.0327
                  TF2            0.1518
german-credit     sklearn        0.0007
                  PYT            0.7615
                  TF2            0.0000
lending-club      sklearn        0.0041
                  PYT            0.9156
                  TF2            0.0254


In [22]:
load_npz(Path("cf_metric_results/raw_vectors/compas-recidivism__TF2__DiCE__50.npz"))['robustness']


array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

in_path  = Path("cf_metric_results") / "summary_metrics_all.csv"
out_path = Path("cf_metric_results") / "summary_metrics_all_trimmed.csv"

df = pd.read_csv(in_path)

def round_sig(x, sig=2):
    if pd.isna(x):
        return x
    return float(f"{abs(x):.{sig}g}")

num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    df[col] = df[col].abs().apply(round_sig)

df.to_csv(out_path, index=False)

out_path


PosixPath('cf_metric_results/summary_metrics_all_trimmed.csv')

In [9]:

# --------------------------------------------------
# paths
# --------------------------------------------------
IN_PATH  = Path("cf_metric_pvals.csv")
OUT_PATH = Path("cf_metric_pvals_clipped.csv")

# --------------------------------------------------
# load and clean
# --------------------------------------------------
df = pd.read_csv(IN_PATH)

# identify numeric cols (floats or ints)
num_cols = df.select_dtypes(include=["number"]).columns

# 1) clip negatives → take abs
df[num_cols] = df[num_cols].abs()

# 2) round to 2 significant figures
df[num_cols] = (
    df[num_cols]
    .apply(lambda col: np.vectorize(lambda x: float(f"{x:.2g}"))(col.values))
)

# --------------------------------------------------
# save
# --------------------------------------------------
df.to_csv(OUT_PATH, index=False)
print(f"✔ saved cleaned p-values → {OUT_PATH}")

✔ saved cleaned p-values → cf_metric_pvals_clipped.csv
